In [1]:
import os    # путь к файлу
import zstandard as zstd    # декомпрессор
import tarfile    # распаковка архива
import glob    # поиск файлов по шаблону

from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd

from joblib import Parallel, delayed, cpu_count    # распараллеливание

import math

In [2]:
#### для компьютера
path = '.'

# Распаковываем архив с масками

In [3]:
# def extract_with_dictionary(tar_file_path, dictionary_path, extract_path):
#     # Создаем декомпрессор с пользовательским словарем
#     with open(dictionary_path, 'rb') as dict_file:
#         dictionary = zstd.ZstdCompressionDict(dict_file.read())

#     dctx = zstd.ZstdDecompressor(dict_data=dictionary)

#     # Открываем и распаковываем tar-архив
#     with open(tar_file_path, 'rb') as compressed_file:
#         with dctx.stream_reader(compressed_file) as decompressed_stream:
#             with tarfile.open(fileobj=decompressed_stream, mode='r|') as tar:
#                 tar.extractall(path=extract_path)


# # полные пути к файлам
# tar_file_path = os.path.join(path, 'mask_acyсl.tar.zst')
# dictionary_path = os.path.join(path, 'dictionary')

# # извлекаем файлы в ./mask_acycl
# extract_with_dictionary(tar_file_path, dictionary_path, path + '/mask_acycl_2d')

### Проверяем правильность распаковки архива

In [4]:
# def quick_analysis(directory_path):
#     """Быстрый анализ файлов"""
#     print(directory_path)

#     files = glob.glob(os.path.join(directory_path, "mask_acycl_*_*_*_*_*"))

#     print(f"Общее количество файлов: {len(files)}")

#     levels, years, months, days, times = set(), set(), set(), set(), set()

#     for file_path in files:
#         filename = os.path.basename(file_path)
#         parts = filename.split('_')
#         if len(parts) >= 7:
#             levels.add(parts[2])
#             years.add(parts[3])
#             months.add(parts[4])
#             days.add(parts[5])
#             times.add(parts[6])

#     print(f"Уровни: {sorted(levels)}")
#     print(f"Года: {sorted(years)}")
#     print(f"Месяцы: {sorted(months)}")
#     print(f"Дни: {sorted(days)}")
#     print(f"Время: {sorted(times)}")


# quick_analysis(path + '/mask_acycl_2d')

In [5]:
# количество файлов, которое должно быть в дииректории
# (11 * 366 + 34 * 365) * 4 * 7

# Вспомогательные функции

### Названия файлов

In [6]:
def create_filename_2d(dt, level):
    """
    Создает название файла по datetime и уровню

    Parameters:
    dt (datetime): объект datetime
    level (int): уровень (например, 1000, 850, 500)

    Returns:
    str: название файла в формате mask_cycl_level_year_month_day_hour.npy
    """
    # Форматируем компоненты даты с ведущими нулями где необходимо
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour

    # Создаем название файла
    filename = f"mask_acycl_{level}_{year}_{month:02d}_{day:02d}_{hour:02d}.npy"

    return filename

In [7]:
def create_filename_3d(dt):
    """
    Создает название файла по datetime

    Parameters:
    dt (datetime): объект datetime

    Returns:
    str: название файла в формате mask_acycl_year_month_day_hour.npy
    """
    # Форматируем компоненты даты с ведущими нулями где необходимо
    year = dt.year
    month = dt.month
    day = dt.day
    hour = dt.hour

    # Создаем название файла
    filename = f"mask_acycl_{year}_{month:02d}_{day:02d}_{hour:02d}.npy"

    return filename

### Маски

In [8]:
def create_bit_masks(arr):
    """
    Самая быстрая версия - создаем все маски сразу.
    """
    unique_values = np.unique(arr)
    return {val: (arr == val).astype(np.uint8) for val in unique_values if val != 0}

In [9]:
# def create_2d_mask(bit_masks_dict):
#     """
#     Быстрое преобразование словаря битовых масок в 2D маску.
#     Использует векторные операции NumPy для ускорения.
    
#     Args:
#         bit_masks_dict: словарь, где ключи - номера объектов, значения - бинарные маски
    
#     Returns:
#         2D маска, где каждый пиксель содержит номер объекта или 0 (фон)
#     """
#     # if not bit_masks_dict:
#     #     # Если словарь пустой, возвращаем нулевую маску подходящего размера
#     #     sample_mask = next(iter(bit_masks_dict.values())) if bit_masks_dict else None
#     #     if sample_mask is not None:
#     #         return np.zeros_like(sample_mask, dtype=np.int32)
#     #     else:
#     #         return np.array([[0]], dtype=np.int32)
    
#     # Получаем размер маски из первого элемента
#     first_mask = next(iter(bit_masks_dict.values()))
#     result_mask = np.zeros_like(first_mask, dtype=np.int32)
    
#     # Векторная операция: сумма (маска * ключ) для всех элементов
#     for key, bit_mask in bit_masks_dict.items():
#         result_mask += bit_mask.astype(np.int32) * key
    
#     return result_mask

In [10]:
def create_2d_mask(bit_masks):

    if len(bit_masks) == 0:
        print('РЕАЛЬНО')
        return np.zeros((37, 144)) # None

    first_mask = next(iter(bit_masks.values()))

    res_mask = np.zeros_like(first_mask)

    for key, mask in bit_masks.items():
        res_mask[mask == 1] = key

    return res_mask

### Графики-картинки

In [11]:
def plot_problem_cyclones(t, l_bottom, l_top, # время и уровни
                          bottom_mask, top_mask, # маски с проблемными циклонами
                          problem_bottom_cyclones, problem_top_cyclones, # номера проблемных циклонов
                          intersection_area, # площадь пересечения (одинакова для всех проблемных пар циклонов)
                          filename_suffix=""):
    """
    Визуализирует проблемные циклоны на двух уровнях
    
    Parameters:
    t - временная метка
    l_bottom, l_top - названия уровней
    bottom_mask, top_mask - 2D маски для визуализации
    problem_bottom_cyclones, problem_top_cyclones - списки проблемных циклонов
    intersection_area - площадь пересечения (одинаковая для всех проблемных пар)
    filename_suffix - дополнительная информация для имени файла
    """
    
    # Создаем график
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Функция для создания заголовка с информацией о циклоне
    def create_cyclone_title(cyclone_id, mask):
        area = np.sum(mask == cyclone_id)
        intersection_percentage = (intersection_area / area) * 100
        return f'Cyclone {cyclone_id}: area={area} cells VS. intersection={intersection_area} cells ({intersection_percentage:.1f}%)'
    
    # Верхний график - l_top
    sns.heatmap(top_mask, ax=ax1, cmap='tab10', cbar_kws={'label': 'Cyclone ID'})
    ax1.invert_yaxis()
    
    # Создаем заголовки для всех проблемных верхних циклонов
    top_titles = []
    for cyclone_id in problem_top_cyclones:
        top_titles.append(create_cyclone_title(cyclone_id, top_mask))
    
    ax1.set_title('\n'.join(top_titles))
    ax1.set_ylabel(f'{l_top} hPa')
    
    # Нижний график - l_bottom
    sns.heatmap(bottom_mask, ax=ax2, cmap='tab10', cbar_kws={'label': 'Cyclone ID'})
    ax2.invert_yaxis()
    
    # Создаем заголовки для всех проблемных нижних циклонов
    bottom_titles = []
    for cyclone_id in problem_bottom_cyclones:
        bottom_titles.append(create_cyclone_title(cyclone_id, bottom_mask))
    
    ax2.set_title('\n\n'.join(bottom_titles))
    ax2.set_ylabel(f'{l_bottom} hPa')
    
    # Общий заголовок
    fig.suptitle(f'{t}', 
                fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    # Сохраняем картинку
    filename = f'warning_{t.strftime("%Y%m%d_%H%M")}_{l_bottom}to{l_top}{filename_suffix}.png'
    plt.savefig(path + '/warning_pictures_modern' + '/' + filename, dpi=150, bbox_inches='tight')
    plt.close()
    
    return filename

### Поправка на сетку

In [12]:
def band_area_ratios(latitudes, degrees=True):

    if degrees:
        latitudes = [math.radians(x) for x in latitudes]

    areas = [
        math.sin(latitudes[i+1]) - math.sin(latitudes[i])
        for i in range(len(latitudes)-1)
    ]

    polar_area = areas[-1]

    return [a / polar_area for a in areas]

In [13]:
# размер оригинальной сетки - 37 x 144
# np.load(path + '/mask_cycl_2d' + '/' + create_filename_2d(datetime(2023, 12, 31, 18, 0), 300)).shape

In [14]:
# оригинальная сетка (центры полос) - от 0 до 90, шаг 2.5 градуса, 37 элементов
# lats = []
# for i in range(0, 900+1, 25):
#     lats.append(i / 10)
# print(lats)
# print(len(lats))

In [15]:
# границы полос
lats = []
lats.append(0.0)
for i in range(125, 9000, 250):
    lats.append(i / 100)
lats.append(90.0)
print(lats)
print(len(lats))

[0.0, 1.25, 3.75, 6.25, 8.75, 11.25, 13.75, 16.25, 18.75, 21.25, 23.75, 26.25, 28.75, 31.25, 33.75, 36.25, 38.75, 41.25, 43.75, 46.25, 48.75, 51.25, 53.75, 56.25, 58.75, 61.25, 63.75, 66.25, 68.75, 71.25, 73.75, 76.25, 78.75, 81.25, 83.75, 86.25, 88.75, 90.0]
38


In [16]:
ratios = band_area_ratios(lats)
print(ratios)
print(len(ratios))

[91.66961108949424, 183.1647238641195, 182.6415610869729, 181.77072971762945, 180.55388743309558, 178.9933505618209, 177.09208967443465, 174.8537239290965, 172.28251418222357, 169.3833548777069, 166.16176473005982, 162.62387621923156, 158.77642391708343, 154.6267316677468, 150.18269864627217, 145.4527843220976, 140.44599235596965, 135.17185346096448, 129.6404072602294, 123.86218317599676, 117.84818038622107, 111.60984688702862, 105.15905770079698, 98.50809227138188, 91.66961108949455, 84.65663159273764, 77.48250338617558, 70.16088283060101, 62.70570704687417, 55.13116738582483, 47.45168241420453, 39.68187046813227, 31.836521826253694, 23.930570555609737, 15.979066083787407, 7.997144551484985, 1.0]
37


In [17]:
ratios_vector = np.array(ratios[::-1])
ratios_vector

array([  1.        ,   7.99714455,  15.97906608,  23.93057056,
        31.83652183,  39.68187047,  47.45168241,  55.13116739,
        62.70570705,  70.16088283,  77.48250339,  84.65663159,
        91.66961109,  98.50809227, 105.1590577 , 111.60984689,
       117.84818039, 123.86218318, 129.64040726, 135.17185346,
       140.44599236, 145.45278432, 150.18269865, 154.62673167,
       158.77642392, 162.62387622, 166.16176473, 169.38335488,
       172.28251418, 174.85372393, 177.09208967, 178.99335056,
       180.55388743, 181.77072972, 182.64156109, 183.16472386,
        91.66961109])

In [18]:
ratios_matrix = np.tile(ratios_vector.reshape(-1, 1), (1, 144))
ratios_matrix

array([[  1.        ,   1.        ,   1.        , ...,   1.        ,
          1.        ,   1.        ],
       [  7.99714455,   7.99714455,   7.99714455, ...,   7.99714455,
          7.99714455,   7.99714455],
       [ 15.97906608,  15.97906608,  15.97906608, ...,  15.97906608,
         15.97906608,  15.97906608],
       ...,
       [182.64156109, 182.64156109, 182.64156109, ..., 182.64156109,
        182.64156109, 182.64156109],
       [183.16472386, 183.16472386, 183.16472386, ..., 183.16472386,
        183.16472386, 183.16472386],
       [ 91.66961109,  91.66961109,  91.66961109, ...,  91.66961109,
         91.66961109,  91.66961109]], shape=(37, 144))

### Собственно преобразование масок

In [19]:
levels = [1000, 925, 850, 700, 600, 500, 300]    #### [300, 500, 600, 700, 850, 925, 1000]

In [20]:
start_time = datetime(1979, 1, 1, 0, 0)    #### datetime(1979, 1, 1, 0, 0)
end_time = datetime(2023, 12, 31, 18, 0)    #### datetime(2023, 12, 31, 18, 0)
time_delta = timedelta(hours=6)

times = []
t = start_time
while t <= end_time:
    times.append(t)
    t += time_delta

In [21]:
def process_datetime_ver2(t):
    l_bottom = levels[0]
    mask_bottom = np.load(path + '/mask_acycl_2d' + '/' + create_filename_2d(t, l_bottom))
    bit_masks_bottom = create_bit_masks(mask_bottom[0])

    res_3d_mask = mask_bottom # сохраняем самый нижний уровень неизменным

    for i in range (1, len(levels)):    #### 
        l_top = levels[i]
        mask_top = np.load(path + '/mask_acycl_2d' + '/' + create_filename_2d(t, l_top))[0]
        if np.max(mask_top) == 0:
            print("No cyclones:", t, l_top)
        # else:
        #     print('Cyclones')
        
        bottom_unique_keys = sorted(bit_masks_bottom.keys())    #np.unique(bit_masks_bottom.keys())[0]
        top_unique_keys = sorted(int(k) for k in np.unique(mask_top) if k != 0)    #np.unique(bit_masks_top.keys())[0]

        intersection_df = pd.DataFrame(index=bottom_unique_keys, columns=top_unique_keys, dtype=float)
        for bottom_key in bottom_unique_keys:
            mask_intersection = np.multiply(bit_masks_bottom.get(bottom_key), mask_top)


            bit_masks_intersection = create_bit_masks(mask_intersection)

            intersection_unique_keys = sorted(bit_masks_intersection.keys())

            for intersection_key in intersection_unique_keys:
                # intersection_res = np.sum(bit_masks_intersection.get(intersection_key))
                intersection_res = np.sum(bit_masks_intersection.get(intersection_key) * ratios_matrix) # ПОПРАВКА НА ШИРОТУ
                intersection_df.loc[bottom_key, intersection_key] = intersection_res

        # Создаем список пар: номер объекта из mask_top + варианты переименования
        rename_options = {}
        for top_key in top_unique_keys:
            rename_options[top_key] = []  # изначально все варианты пустые

        # Заполняем список по правилам
        for bottom_key in bottom_unique_keys:
            row = intersection_df.loc[bottom_key]
            non_zero_values = row[row > 0]
            
            if len(non_zero_values) == 0:
                # Правило 1: если в строке все нули, ничего не делаем
                # print(bottom_key, '->')
                continue    # pass
            elif len(non_zero_values) == 1:
                # Правило 2: если единственное ненулевое значение
                # print(bottom_key, '->', non_zero_values)
                top_key = non_zero_values.index[0]
                rename_options[top_key].append(bottom_key)
            else:
                # Правило 3: если несколько ненулевых значений
                max_value = non_zero_values.max()
                max_columns = non_zero_values[non_zero_values == max_value].index
                
                if len(max_columns) == 1:
                    # Один максимальный - добавляем его
                    # print(bottom_key, '->', max_columns, '(max)')
                    top_key = max_columns[0]
                    rename_options[top_key].append(bottom_key)
                else:
                    # ПРЕДВАРИТЕЛЬНО: ведём себя так, пересечений нет (в строке все нули)

                    # Несколько максимальных - предупреждение
                    print(f'Warning: {t}, {l_bottom} -> {l_top}')
                    print(f'         Нельзя определить КУДА переходит циклон {bottom_key}, потому что несколько одинаковых вариантов: {list(max_columns)}')
                    
                    continue    # pass
                    
                    
                    # # Создаем маски только для проблемных циклонов
                    # problem_bottom_dict = {bottom_key: bit_masks_bottom[bottom_key]}
                    # bottom_mask_vis = create_2d_mask(problem_bottom_dict)
                    
                    # top_mask_vis = np.where(np.isin(mask_top, max_columns), mask_top, 0)
                    
                    # # Получаем площадь пересечения (одинаковая для всех вариантов)
                    # intersection_area = intersection_df.loc[bottom_key, max_columns[0]]
                    
                    # # Сохраняем картинку с проблемными циклонами
                    # filename = plot_problem_cyclones(
                    #     t=t,
                    #     l_bottom=l_bottom,
                    #     l_top=l_top,
                    #     bottom_mask=bottom_mask_vis,
                    #     top_mask=top_mask_vis,
                    #     problem_bottom_cyclones=[bottom_key],
                    #     problem_top_cyclones=list(max_columns),
                    #     intersection_area=intersection_area,
                    #     filename_suffix=f"_cyclone{bottom_key}"
                    # )

                    
                    


        max_cycl_name = max(bit_masks_bottom.keys())

        # Создаем новый словарь для переименованных масок
        new_bit_masks_top = {}

        # Обрабатываем каждый объект из верхнего уровня
        for top_key in top_unique_keys:
            possible_renames = rename_options[top_key]
        
            if len(possible_renames) == 0:
                # 1. Если варианта переименования нет
                new_name = max_cycl_name + 1
                new_bit_masks_top[new_name] = (mask_top == top_key).astype(np.uint8)
                max_cycl_name += 1
        
            elif len(possible_renames) == 1:
                # 2. Если единственный вариант переименования
                new_name = possible_renames[0]
                new_bit_masks_top[new_name] = (mask_top == top_key).astype(np.uint8)
        
            else:
                # 3. Если вариантов переименования несколько
                max_intersection = -1
                best_bottom_keys = []
        
                for bottom_key in possible_renames:
                    intersection_value = intersection_df.loc[bottom_key, top_key]
                    if intersection_value > max_intersection:
                        max_intersection = intersection_value
                        best_bottom_keys = [bottom_key]
                    elif intersection_value == max_intersection:
                        best_bottom_keys.append(bottom_key)
        
                if len(best_bottom_keys) == 1:
                    new_name = best_bottom_keys[0]
                    new_bit_masks_top[new_name] = (mask_top == top_key).astype(np.uint8)
                else:
                    # ПРЕДВАРИТЕЛЬНО: как будто пересечений нет
                    new_name = max_cycl_name + 1
                    new_bit_masks_top[new_name] = (mask_top == top_key).astype(np.uint8)
                    max_cycl_name += 1

                    # Несколько максимальных - предупреждение
                    print(f'Warning: {t}, {l_bottom} -> {l_top}')
                    print(f'         Нельзя определить ОТКУДА приходит циклон {top_key}, потому что несколько одинаковых вариантов: {best_bottom_keys}')

        
                    # # Создаем маски только для проблемных циклонов
                    # problem_bottom_dict = {k: bit_masks_bottom[k] for k in best_bottom_keys}
                    # bottom_mask_vis = create_2d_mask(problem_bottom_dict)

                    # top_mask_vis = np.where(mask_top == top_key, mask_top, 0)
                    
                    # # Получаем площадь пересечения (одинаковая для всех вариантов)
                    # intersection_area = intersection_df.loc[best_bottom_keys[0], top_key]
                    
                    # # Сохраняем картинку с проблемными циклонами
                    # filename = plot_problem_cyclones(
                    #     t=t,
                    #     l_bottom=l_bottom,
                    #     l_top=l_top,
                    #     bottom_mask=bottom_mask_vis,
                    #     top_mask=top_mask_vis,
                    #     problem_bottom_cyclones=best_bottom_keys,
                    #     problem_top_cyclones=[top_key],
                    #     intersection_area=intersection_area,
                    #     filename_suffix=f"_cyclone{top_key}"
                    # )

                    
                    
        # print(new_bit_masks_top)

        new_mask_top = create_2d_mask(new_bit_masks_top)

        l_bottom = l_top
        bit_masks_bottom = new_bit_masks_top

        # print("mask_top shape:", mask_top.shape)
        # print("new_mask_top shape:", new_mask_top.shape)

        # Исправление: добавляем новую ось и объединяем массивы
        new_mask_top_expanded = np.expand_dims(new_mask_top, axis=0)  # добавляем ось уровня
        res_3d_mask = np.concatenate((res_3d_mask, new_mask_top_expanded), axis=0)

    np.save(path + '/mask_acycl_3d' + '/' + create_filename_3d(t), res_3d_mask)

    return t    # для контроля отработанных временных меток

In [22]:
# start = datetime.now()
# print('We started!')
# print(start)
# print()

# for t in times:
#     process_datetime_ver2(t)

#     if ((t+time_delta).month == 1) and ((t+time_delta).day == 1) and ((t+time_delta).hour == 0):
#         print(t.year, 'finished')
#         current_time = datetime.now()
#         print(current_time)

# finish = datetime.now()
# print()
# print('We finished!')
# print(finish)

# programm_length = (finish - start).total_seconds()
# print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

In [23]:
# start = datetime.now()
# print(start)

# for t in times:
#     process_datetime(t)

# finish = datetime.now()
# print(finish)

# programm_length = (finish - start).total_seconds()
# print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

In [24]:
# start = datetime.now()
# print('We started!')
# print(start)
# print()

# for t in times:
#     process_datetime_ver2(t)

#     if ((t+time_delta).month == 1) and ((t+time_delta).day == 1) and ((t+time_delta).hour == 0):
#         print(t.year, 'finished')
#         current_time = datetime.now()
#         print(current_time)

# finish = datetime.now()
# print()
# print('We finished!')
# print(finish)

# programm_length = (finish - start).total_seconds()
# print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

### С распараллеливанием

In [25]:
n_jobs = 1 # cpu_count()  # оставить 1 ядро системе
print(n_jobs)

1


In [26]:
start = datetime.now()
print(start)

Parallel(n_jobs=n_jobs)(delayed(process_datetime_ver2)(t) for t in times)

finish = datetime.now()
print(finish)

programm_length = (finish - start).total_seconds()
print(f'{programm_length // 3600} hours {programm_length % 3600 // 60} minutes {programm_length % 3600 % 60} seconds')

2026-03-05 17:41:21.342851
No cyclones: 1979-01-02 18:00:00 300
РЕАЛЬНО
         Нельзя определить ОТКУДА приходит циклон 7, потому что несколько одинаковых вариантов: [np.uint8(12), np.uint8(13)]
         Нельзя определить ОТКУДА приходит циклон 3, потому что несколько одинаковых вариантов: [np.uint8(8), np.uint8(9)]
No cyclones: 1979-02-07 18:00:00 300
РЕАЛЬНО
No cyclones: 1979-02-09 00:00:00 300
РЕАЛЬНО
No cyclones: 1979-02-10 00:00:00 300
РЕАЛЬНО
No cyclones: 1979-02-10 06:00:00 300
РЕАЛЬНО
No cyclones: 1979-02-15 06:00:00 300
РЕАЛЬНО
No cyclones: 1979-02-15 12:00:00 300
РЕАЛЬНО
         Нельзя определить ОТКУДА приходит циклон 18, потому что несколько одинаковых вариантов: [np.uint8(28), np.uint8(30)]
         Нельзя определить ОТКУДА приходит циклон 22, потому что несколько одинаковых вариантов: [np.uint8(26), np.uint8(30)]
         Нельзя определить ОТКУДА приходит циклон 7, потому что несколько одинаковых вариантов: [np.uint8(12), np.uint8(13)]
         Нельзя определить КУДА п

ValueError: max() arg is an empty sequence